In [1]:
import pandas as pd
import optuna
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from sklearn.preprocessing import MaxAbsScaler

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor,
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor

from utils import *


optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
# Suppress pandas warnings
warnings.simplefilter(action='ignore', category=FutureWarning)



/home/khaled/scraper/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
versions = [
            'classic',
            'stations_basic', 
            'stations_weighted_all',
            #'stations_weights_inverse',
            #'stations_weights_gaussian',
            #'stations_weights_logarithmic',
            #'stations_weights_linear'
                ]

n_trials= 50
random_state = 42
test_size = 0.2

model_classes = [
    # slow models
    RandomForestRegressor,
    GradientBoostingRegressor,
    #AdaBoostRegressor, Not going to use this one
    #MLPRegressor, too slow (maybe try with gpu support using pytorch or tensorflow)
    
    # medium models
    SVR,
    
    # fast models
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet,
    KNeighborsRegressor,
    DecisionTreeRegressor,
    
]

In [3]:
df_scores = pd.DataFrame(columns=['version', 'model', 'r2', 'mse', 'mae', 'best_params', 'validation_mae'])
studies = {}

In [4]:
for version in versions:
    for model_class in model_classes:
        print("="*50)
        print(f'{version} - {model_class.__name__}')
        X_train, X_test, y_train, y_test  = prepare_data(version=version, test_size=test_size, random_state=random_state)
        # Define the metric function for optimization
        metric_function = mean_absolute_error
        # Create the objective function with additional parameters
        objective = create_objective_function(X_train, y_train, metric_function, model_class)
        study = optuna.create_study(direction="minimize", study_name=f"{version}_{model_class.__name__}")
        
        # Optimize the study, use more trials for better performance
        if model_class == LinearRegression: # No hyperparameters to tune
            study.optimize(objective, n_trials=1, show_progress_bar=True,gc_after_trial=True) 
        else:      
            study.optimize(objective, n_trials=n_trials, show_progress_bar=True,gc_after_trial=True) 
            
        # train the model with the best parameters
        if model_class in [SVR, KNeighborsRegressor, LinearRegression]:
             model = model_class(**study.best_params)
        else:
            model = model_class(**study.best_params, random_state=random_state)
            
        scaler = MaxAbsScaler()
        # Fit the scaler on the training set and transform the training and testing sets
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
            
        model.fit(X_train_scaled, y_train)
        
        # evaluate the model
        y_pred = model.predict(X_test_scaled)
        r2 = r2_score(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        validation_mae = study.best_value
        new_row = {'version': version, 'model': model_class.__name__, 'r2': r2, 'mse': mse, 'mae': mae, 'best_params': study.best_params, 'validation_mae': validation_mae}
        df_scores = pd.concat([df_scores, pd.DataFrame([new_row])], ignore_index=True)
        # save the study
        studies[f'{version}_{model_class.__name__}'] = study

        

classic - RandomForestRegressor


  0%|          | 0/1 [00:00<?, ?it/s]

Best trial: 0. Best value: 236.075: 100%|██████████| 1/1 [00:01<00:00,  1.91s/it]


classic - GradientBoostingRegressor


Best trial: 0. Best value: 140.695: 100%|██████████| 1/1 [00:11<00:00, 11.38s/it]


...

Best trial: 0. Best value: 280.425: 100%|██████████| 1/1 [00:00<00:00,  2.56it/s]


stations_weighted_all - KNeighborsRegressor


Best trial: 0. Best value: 210.978: 100%|██████████| 1/1 [00:01<00:00,  1.70s/it]


stations_weighted_all - DecisionTreeRegressor


Best trial: 0. Best value: 228.431: 100%|██████████| 1/1 [00:00<00:00,  1.88it/s]


In [5]:
df_scores.sort_values(by='mae', ascending=True, inplace=False).head()


,version,model,r2,mse,mae,best_params,validation_mae
1,classic,GradientBoostingRegressor,0.726785,39395.250138,123.398639,"{'n_estimators': 1500, 'learning_rate': 0.1488...",140.695315
19,stations_weighted_all,GradientBoostingRegressor,0.619088,54924.208710,127.750559,"{'n_estimators': 1100, 'learning_rate': 0.4773...",140.948188
9,stations_basic,RandomForestRegressor,0.737055,37914.426862,127.877672,"{'n_estimators': 500, 'max_depth': 24, 'min_sa...",147.680538
18,stations_weighted_all,RandomForestRegressor,0.730828,38812.320076,130.901248,"{'n_estimators': 800, 'max_depth': 20, 'min_sa...",152.513134
10,stations_basic,GradientBoostingRegressor,0.691486,44485.079832,134.818971,"{'n_estimators': 200, 'learning_rate': 0.45780...",160.087121


In [26]:
# save studies object
add_name = '_'.join(versions)
import pickle
with open(f'studies/studies_{add_name}.pkl', 'wb') as f:
    pickle.dump(studies, f)
# save results
df_scores.to_csv(f'results/results_{add_name}.csv', index=False)

In [23]:
# load studies object

#studies = pickle.load(open(f'studies_{add_name}.pkl', 'rb'))